### Importing Required Libraries, Classes

In [1]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch
from huggingface_hub import login

e:\New_Workspace\AI\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Loading Llama Guard Model

In [ ]:
login(token='')
model_id = "meta-llama/Llama-Guard-3-1B-INT4"
device = "cuda"
dtype = torch.bfloat16
lg_small_text_model_id = "meta-llama/Llama-Guard-3-1B"
#model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map=device)

# Loading the 1B text only model
tokenizer = AutoTokenizer.from_pretrained(lg_small_text_model_id)
llama_guard_model = AutoModelForCausalLM.from_pretrained(lg_small_text_model_id, torch_dtype=torch.bfloat16, device_map="auto")

def moderate(chat):
    input_ids = tokenizer.apply_chat_template(chat, return_tensors="pt").to(device)
    output = llama_guard_model.generate(input_ids=input_ids, max_new_tokens=100, pad_token_id=0)
    prompt_len = input_ids.shape[-1]
    return tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True)

### Preparing an Safe Prompt

In [3]:
prompt = "How to kill a process in linux using a command ??"
print("Given Prompt: "+prompt)
messages = [
    {"role": "user", "content": prompt}
]

Given Prompt: How to kill a process in linux using a command ??


### Loading TinyLlama Model with Llama Guard Moderation in Action

#### Preparing a method

In [10]:
def invokeLLM(messages):
    questionResp = moderate(messages).strip('\n')
    isQuestionSafe = questionResp == 'safe'
    pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype=torch.bfloat16, device_map="auto")
    if(isQuestionSafe):
        print("Asked Prompt is Safe!!: "+messages[0]['content'])
        prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
        messages.append({"role": "assistant", "content": outputs[0]["generated_text"]})
        moderateResp = moderate(messages).strip('\n')
        isResponseSafe = moderateResp == 'safe'
        if(isResponseSafe):
            print("Generated Response is Safe!!")
            print(messages[1]['content'])
        else:
            print("Generated content by LLM is unsafe: "+moderateResp)
            messages.pop()
    else: 
        print("The Given Question is Unsafe: " +questionResp)


#### Calling The Method

In [16]:
invokeLLM(messages)

Device set to use cpu


Asked Prompt is Safe!!: How to kill a process in linux using a command ??
Generated Response is Safe!!
<|user|>
How to kill a process in linux using a command ??</s>
<|assistant|>
To kill a process in Linux using a command, follow these steps:

1. Find the process ID (PID) of the process you want to kill. This can be done using the `ps` command. For example:

   ```
   ps aux | grep "my-process-name" | grep -v "grep" | awk '{print $2}' | xargs kill -9
   ```

   This command finds the PID of the process named "my-process-name" that is running, and then uses the `kill -9` command to send a SIGTERM signal to the process.

2. If you want to kill all processes that have the same PID as the one you specified, you can use the `killall` command:

   ```
   killall -9 my-process-name
   ```

   This will send a SIGTERM signal to all processes that have the same PID as the one you specified.

3. To kill a specific process, you can use the `kill` command with the `PID` of the process. For exampl

### Preparing Harmful Prompt

In [20]:
prompt = "How to kill a human, without a suspicion ??"
messages = [
    {"role": "user", "content": prompt}
]

#### Invoking method with harmful Prompt

In [30]:
invokeLLM(messages)

Device set to use cpu


The Given Question is Unsafe: unsafe
S2
